# P2 langkah 1: evaluasi detektor ayam + VLM pada crop detektor (Colab)

Tujuan: memilih detektor untuk P2 **sebelum** menulis kode layanan. Notebook ini membandingkan, pada 51
frame berlabel (69 ekor):

1. **Detektor**: apakah kotak terbesar jatuh tepat di ayam (IoU ≥ 0,5), berapa ayam tertangkap, berapa
   kotak palsu, dan waktu per frame. Kandidat: YOLO11 COCO kelas `bird` (n dan s) dan detektor custom
   ChickenStress (bila `best.pt` diunggah; kelasnya diabaikan, hanya kotaknya dipakai).
2. **VLM layanan** pada tiga input: frame utuh (seperti sekarang), crop dari anotasi (batas atas, sama
   dengan crop training), dan crop dari detektor terbaik. Crop memakai aturan training persis
   (`scripts/p2_eval.py`, sudah diuji identik piksel dengan crop training).

**Sebelum Run all**
- Runtime → Change runtime type → **T4 GPU**.
- Unggah `Raw Data Chicken Stress.zip` ke panel Files (`/content/`). File ini ada di laptop:
  `E:\Magang DSC\Code - VLM Workflow Kaggle\Raw Data Chicken Stress.zip`. Kalau belum diunggah, sel data
  akan meminta upload.
- Opsional: unggah `best.pt` detektor custom dari
  `E:\Magang DSC\Code - Image Classification Workflow\runs\detect\ChickenStress\config1_stage2_finetune\weights\`.

**Catatan membaca hasil**
- 65 dari 69 ekor berasal dari split *train*, jadi VLM sudah pernah melihat crop-nya. Akurasi absolut
  bukan ukuran data baru; yang bermakna adalah **perbandingan** frame utuh vs crop anotasi vs crop
  detektor pada gambar yang sama.
- Label acuan: `label_akhir` di `docs/manifest/manifest_instance.csv` (label SCS dari anotasi indikator).
  Akurasi dihitung pada ekor berlabel `stress`/`not_stress`; frame utuh dan crop dinilai terhadap ekor
  terbesar, crop detektor terhadap ekor yang paling tertutup kotaknya. Frame tanpa deteksi dihitung salah.
- Ultralytics berlisensi **AGPL-3.0** (termasuk bobot turunannya). Notebook ini hanya evaluasi.

Hasil: `p2_results.zip` (unduh otomatis di sel terakhir). Kirim file itu kembali.

In [ ]:
REPO = "https://github.com/WANGDELL30/agronova-poultry-dashboard"
BRANCH = "fix/web-batch-1"
RAW_ZIP = "/content/Raw Data Chicken Stress.zip"
CUSTOM_WEIGHTS = "/content/best.pt"   # opsional; dilewati kalau tidak ada
DETECTORS = [
    {"name": "coco_yolo11n_bird", "weights": "yolo11n.pt", "classes": [14]},   # COCO 14 = bird
    {"name": "coco_yolo11s_bird", "weights": "yolo11s.pt", "classes": [14]},
    {"name": "chickenstress_custom", "weights": CUSTOM_WEIGHTS, "classes": None},
]
CONFS = [0.05, 0.1, 0.25, 0.5]
RUN_VLM = True   # False = hanya evaluasi detektor (cepat, tanpa memuat VLM)

In [ ]:
!rm -rf /content/repo && git clone -q -b $BRANCH $REPO /content/repo
%cd /content/repo/apps/inference
!pip -q install ultralytics
# Pin layanan dipasang terakhir supaya versi VLM sama dengan yang lolos parity (S10).
!pip -q install -r requirements.txt -r requirements-real.txt
# Colab membawa torchao lama yang bentrok dengan transformers baru; tidak dipakai di sini
!pip -q uninstall -y torchao

In [ ]:
import json, os, sys, time, zipfile
from pathlib import Path

if not Path(RAW_ZIP).exists():
    from google.colab import files
    print("Unggah 'Raw Data Chicken Stress.zip'")
    RAW_ZIP = "/content/" + next(iter(files.upload()))
RAW_DIR = Path("/content/raw")
with zipfile.ZipFile(RAW_ZIP) as z:
    z.extractall(RAW_DIR)
DATA = next(RAW_DIR.rglob("train/_annotations.coco.json")).parent.parent

sys.path.insert(0, "/content/repo/apps/inference")
from scripts.p2_eval import (Box, Detection, crop_like_training, detector_metrics, largest_detection,
                             load_ground_truth, matched_instance, summarize_vlm)

REPO_DIR = Path("/content/repo")
frames = load_ground_truth(REPO_DIR / "results/vlm/pipeline_all_df.csv",
                           REPO_DIR / "docs/manifest/manifest_instance.csv")
frame_path = lambda f: DATA / f.split / f.file_name
missing = [f.file_name for f in frames.values() if not frame_path(f).exists()]
print(len(frames), "frame,", sum(len(f.instances) for f in frames.values()), "ekor, file hilang:", len(missing))
assert not missing, missing[:5]

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

detections, timing = {}, {}
for spec in DETECTORS:
    weights = spec["weights"]
    # A path with a folder must exist; a bare name like "yolo11n.pt" is downloaded by ultralytics.
    if Path(weights).parent != Path(".") and not Path(weights).exists():
        print("lewati", spec["name"], "- bobot tidak ada:", weights)
        continue
    model = YOLO(weights)
    per_frame, seconds = {}, []
    for f in frames.values():
        t0 = time.perf_counter()
        r = model.predict(str(frame_path(f)), conf=min(CONFS), classes=spec["classes"], verbose=False, device=0)[0]
        seconds.append(time.perf_counter() - t0)
        per_frame[f.file_name] = [
            Detection(Box(x1, y1, x2 - x1, y2 - y1), c)
            for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist())
        ]
    detections[spec["name"]] = per_frame
    timing[spec["name"]] = round(1000 * sorted(seconds)[len(seconds) // 2], 1)  # median, tanpa warm-up
    print(spec["name"], "selesai,", timing[spec["name"]], "ms/frame (median)")
    del model
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd

rows = []
for name, per_frame in detections.items():
    for conf in CONFS:
        m = detector_metrics(frames, per_frame, conf)
        rows.append({"detector": name, "ms_per_frame": timing[name], **m})
det_df = pd.DataFrame(rows).sort_values(
    ["largest_hits_largest_chicken", "largest_hits_a_chicken", "false_boxes_per_frame"],
    ascending=[False, False, True],
)
display(det_df)
best = det_df.iloc[0]
BEST = (best.detector, float(best.conf))   # boleh diganti manual, misalnya ("coco_yolo11s_bird", 0.25)
print("Detektor untuk evaluasi VLM:", BEST)
det_df.to_csv("/content/p2_detector_metrics.csv", index=False)

In [ ]:
vlm_rows = []
if RUN_VLM:
    import secrets
    os.environ["INFERENCE_MODE"] = "real"
    # Wajib di mode real (AUDIT S4). Tidak dipakai di sini: layanan HTTP tidak dijalankan.
    os.environ.setdefault("INFERENCE_API_KEY", secrets.token_urlsafe(16))
    try:  # hanya perlu kalau repo adapter private
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
    from app.backends import RealBackend
    from app.config import load_settings
    from app.pipeline import classify_bytes

    backend = RealBackend(load_settings())
    det_name, det_conf = BEST
    for i, f in enumerate(frames.values(), 1):
        largest = f.largest
        inputs = {
            "full": (frame_path(f).read_bytes(), largest.label),
            "gt_crop": (crop_like_training(frame_path(f), largest.box), largest.label),
        }
        top = largest_detection(detections[det_name][f.file_name], det_conf)
        if top is None:
            inputs["det_crop"] = (None, largest.label)   # tidak ada deteksi: dihitung salah
        else:
            match = matched_instance(f, top.box)
            inputs["det_crop"] = (crop_like_training(frame_path(f), top.box), match.label if match else None)
        for kind, (data, gt) in inputs.items():
            row = {"file": f.file_name, "chickens": len(f.instances), "input": kind, "gt_label": gt}
            if data is None:
                row.update(status="no_chicken", pred_label=None)
            else:
                r = classify_bytes(data, backend)
                res = r["result"] or {}
                row.update(status=r["status"], pred_label=res.get("label"), total_ya=res.get("total_ya"),
                           visible=res.get("visible_count"), inference_ms=r["inference_ms"])
            vlm_rows.append(row)
        print(i, end=" ", flush=True)
    print()
    display(pd.DataFrame(summarize_vlm(vlm_rows)).T)

In [ ]:
import importlib.metadata as md
from google.colab import files

versions = {p: md.version(p) for p in ("ultralytics", "torch", "transformers", "peft")}
versions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
summary = {
    "best_detector": BEST,
    "versions": versions,
    "detector_metrics": det_df.to_dict("records"),
    "vlm": summarize_vlm(vlm_rows) if vlm_rows else None,
    "vlm_multi_chicken_frames": summarize_vlm([r for r in vlm_rows if r["chickens"] > 1]) if vlm_rows else None,
    "note": "65/69 ekor dari split train: bandingkan antar-input, jangan baca akurasi absolut.",
}
Path("/content/p2_summary.json").write_text(json.dumps(summary, indent=1, default=str))
pd.DataFrame(vlm_rows).to_csv("/content/p2_vlm_rows.csv", index=False)
with zipfile.ZipFile("/content/p2_results.zip", "w") as z:
    for name in ("p2_summary.json", "p2_detector_metrics.csv", "p2_vlm_rows.csv"):
        z.write(f"/content/{name}", name)
print(json.dumps(summary["vlm"], indent=1, default=str))
files.download("/content/p2_results.zip")